# 🕌 QuranASR — Fine-tune Whisper for Quran Recitation

Fine-tunes `openai/whisper-small` on ~75K Quran recitation samples from 12 reciters.

📖 [GitHub](https://github.com/tialkan/QuranASR) · 🤗 [HuggingFace](https://huggingface.co/baristiran) · 📱 [Tilavet App](https://tilavet.app)

---

## How to run

**Phase 1 — Data prep (CPU + High RAM, no GPU needed):**
1. Run Cell 1 (install) — when it says **"Restart"**, click the button Colab shows you
2. Run Cell 2 → 3 → 4 → 5 → 6 in order

**Phase 2 — Training (switch to A100 GPU + High RAM):**
1. Runtime → Change runtime type → **A100 GPU + High RAM**
2. Run Cell 1 (install) — restart when prompted
3. Run Cell 2, then Cell 7 (training), then Cell 8 (push to HuggingFace)

**If Colab disconnects:** Run Cell 1 → restart → Cell 2 → Cell 7 (resumes from checkpoint)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install packages
#
# After this finishes, Colab will show a "RESTART SESSION" button.
# Click it, then continue from Cell 2.
# ══════════════════════════════════════════════════════════════════════════════
!pip install -q --force-reinstall --no-deps \
    transformers datasets accelerate evaluate jiwer soundfile librosa huggingface_hub
!pip install -q \
    transformers datasets accelerate evaluate jiwer soundfile librosa huggingface_hub
!apt-get install -y -qq ffmpeg > /dev/null 2>&1

print()
print('✅ Done. Now click the RESTART SESSION button above ↑')
print('   Then run Cell 2.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Mount Drive + setup (run after every restart)
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, torch, json, psutil
from pathlib import Path

BASE        = '/content/drive/MyDrive/quran_asr'
MODEL_ID    = 'openai/whisper-small'
OUTPUT_DIR  = f'{BASE}/checkpoints/quran-whisper-small'
DATASET_DIR = f'{BASE}/data/hf_dataset'
AUDIO_DIR   = Path(f'{BASE}/data/audio')
QURAN_FILE  = f'{BASE}/data/quran.json'
MANIFEST    = f'{BASE}/data/manifest.jsonl'
HF_REPO_ID  = 'baristiran/QuranASR'

RECITERS = [
    'Minshawi_Murattal_128kbps', 'Husary_128kbps', 'Alafasy_128kbps',
    'Abdurrahmaan_As-Sudais_192kbps', 'Abdul_Basit_Murattal_192kbps',
    'Mohammad_al_Tablaway_128kbps', 'Maher_AlMuaiqly_128kbps',
    'Hani_Rifai_192kbps', 'Ibrahim_Akhdar_128kbps',
    'Yasser_Ad-Dussary_128kbps', 'Abdullah_Basfar_192kbps',
    'Sahl_Yassin_128kbps',
]

VERSES_PER_SURAH = [
    7,286,200,176,120,165,206,75,129,109,123,111,43,52,99,128,111,110,98,135,
    112,78,118,64,77,227,93,88,69,60,34,30,73,54,45,83,182,88,75,85,54,53,89,
    59,37,35,38,29,18,45,60,49,62,55,78,96,29,22,24,13,14,11,11,18,12,12,30,
    52,52,44,28,28,20,56,40,31,50,40,46,42,29,19,36,25,22,17,19,26,30,20,15,
    21,11,8,8,19,5,8,8,11,11,8,3,9,5,4,7,3,6,3,5,4,5,6,
]

for d in ['data/audio', 'checkpoints']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

# Status
gpu = f'{torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'CPU'
ram = f'{psutil.virtual_memory().total / 1e9:.0f}GB'
has_quran = os.path.exists(QURAN_FILE)
has_manifest = os.path.exists(MANIFEST)
has_dataset = os.path.exists(f'{DATASET_DIR}/train')
n_audio = sum(len(list((AUDIO_DIR/r).glob('*.mp3'))) for r in RECITERS if (AUDIO_DIR/r).exists())

print(f'Device: {gpu} | RAM: {ram}')
print(f'Quran text: {"✅" if has_quran else "❌"}  |  Manifest: {"✅" if has_manifest else "❌"}  |  Dataset: {"✅" if has_dataset else "❌"}  |  Audio: {n_audio:,} files')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Download Quran text from quran.com API (skips if exists)
# ══════════════════════════════════════════════════════════════════════════════
import json, os, time, requests
from tqdm.notebook import tqdm

if os.path.exists(QURAN_FILE):
    suras = json.load(open(QURAN_FILE, encoding='utf-8'))
    print(f'✅ Already on Drive: {sum(len(s["verses"]) for s in suras)} verses')
else:
    print('Downloading Quran text...')
    suras, session = [], requests.Session()
    for sn in tqdm(range(1, 115), desc='Surahs'):
        url = f'https://api.quran.com/api/v4/verses/by_chapter/{sn}?language=ar&words=false&per_page=300&fields=text_uthmani'
        for attempt in range(5):
            try:
                r = session.get(url, timeout=30); r.raise_for_status(); break
            except Exception:
                if attempt == 4: raise
                time.sleep(2 ** attempt)
        verses = [{'id': int(v['verse_key'].split(':')[1]), 'text': v['text_uthmani']}
                  for v in r.json()['verses']]
        suras.append({'id': sn, 'verses': verses})
        time.sleep(0.15)
    with open(QURAN_FILE, 'w', encoding='utf-8') as f:
        json.dump(suras, f, ensure_ascii=False)
    print(f'✅ Downloaded {sum(len(s["verses"]) for s in suras)} verses')

suras = json.load(open(QURAN_FILE, encoding='utf-8'))
print(f'Sample 1:1 → {suras[0]["verses"][0]["text"]}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Download audio from EveryAyah.com (~50-70 GB)
# Resumes automatically — already downloaded files are skipped.
# First run takes hours. Re-running is fast (just verifies).
# ══════════════════════════════════════════════════════════════════════════════
import time, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def dl(reciter, surah, verse):
    dest = AUDIO_DIR / reciter / f'{surah:03d}_{verse:03d}.mp3'
    if dest.exists() and dest.stat().st_size > 0: return 'skip'
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://everyayah.com/data/{reciter}/{surah:03d}{verse:03d}.mp3'
    for a in range(3):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 404: return 'miss'
            r.raise_for_status(); dest.write_bytes(r.content); return 'ok'
        except Exception:
            if a < 2: time.sleep(2**a)
    return 'fail'

tasks = [(r,s,v) for r in RECITERS for s in range(1,115)
         for v in range(1, VERSES_PER_SURAH[s-1]+1)]
print(f'{len(tasks):,} files to check')

ok=skip=miss=fail=0
with ThreadPoolExecutor(max_workers=16) as ex:
    futs = {ex.submit(dl,*t):t for t in tasks}
    with tqdm(total=len(tasks), unit='file') as bar:
        for f in as_completed(futs):
            r = f.result()
            if r=='ok': ok+=1
            elif r=='skip': skip+=1
            elif r=='miss': miss+=1
            else: fail+=1
            bar.update(1); bar.set_postfix(new=ok, exist=skip, miss=miss, fail=fail)
print(f'\n✅ New:{ok:,}  Existing:{skip:,}  NotFound:{miss:,}  Failed:{fail:,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Build manifest (skips if exists)
# Maps each audio file → diacritics-stripped Quran verse text
# ══════════════════════════════════════════════════════════════════════════════
import json, os

if os.path.exists(MANIFEST):
    with open(MANIFEST) as f: count = sum(1 for _ in f)
    print(f'✅ Already on Drive: {count:,} records')
else:
    DIACRITICS = [(0x0610,0x061A),(0x064B,0x065F),(0x06D6,0x06DC),(0x06DF,0x06E8),(0x06EA,0x06ED)]
    def strip(t):
        return ' '.join(''.join(c for c in t if not any(lo<=ord(c)<=hi for lo,hi in DIACRITICS)).split())

    data = json.load(open(QURAN_FILE, encoding='utf-8'))
    records = []
    for sura in data:
        for v in sura['verses']:
            text = strip(v['text'])
            for rec in RECITERS:
                fp = AUDIO_DIR/rec/f"{sura['id']:03d}_{v['id']:03d}.mp3"
                if fp.exists() and fp.stat().st_size > 1000:
                    records.append({'audio_path':str(fp),'text':text,
                                    'surah_id':sura['id'],'verse_id':v['id'],'reciter':rec})
    with open(MANIFEST,'w',encoding='utf-8') as f:
        for r in records: f.write(json.dumps(r,ensure_ascii=False)+'\n')
    print(f'✅ {len(records):,} records written')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Create HF Dataset (lightweight: path + text only)
# ══════════════════════════════════════════════════════════════════════════════
import json, os, numpy as np
from datasets import Dataset, load_from_disk

if os.path.exists(f'{DATASET_DIR}/train') and os.path.exists(f'{DATASET_DIR}/val'):
    train_ds = load_from_disk(f'{DATASET_DIR}/train')
    val_ds   = load_from_disk(f'{DATASET_DIR}/val')
    print(f'✅ From Drive → Train:{len(train_ds):,}  Val:{len(val_ds):,}')
else:
    with open(MANIFEST, encoding='utf-8') as f:
        records = [json.loads(l) for l in f]
    rng = np.random.default_rng(42)
    idx = rng.permutation(len(records)).tolist()
    n_val = max(500, int(len(records)*0.05))
    vs = set(idx[:n_val])
    train_ds = Dataset.from_list([{'audio_path':r['audio_path'],'text':r['text']} for i,r in enumerate(records) if i not in vs])
    val_ds   = Dataset.from_list([{'audio_path':r['audio_path'],'text':r['text']} for i,r in enumerate(records) if i in vs])
    train_ds.save_to_disk(f'{DATASET_DIR}/train')
    val_ds.save_to_disk(f'{DATASET_DIR}/val')
    print(f'✅ Saved → Train:{len(train_ds):,}  Val:{len(val_ds):,}')

print()
print('='*60)
print('DATA PREP COMPLETE')
print()
print('Now switch runtime:')
print('  Runtime → Change runtime type → A100 GPU + High RAM')
print('  Then: Cell 1 → restart → Cell 2 → Cell 7')
print('='*60)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — TRAINING (requires A100 GPU)
# ══════════════════════════════════════════════════════════════════════════════
import torch, evaluate, glob, threading, time, gc
import soundfile as sf, numpy as np
from dataclasses import dataclass, field
from typing import Any, List, Dict
from datasets import load_from_disk
import transformers
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback, TrainerCallback,
)

# ── Keep Colab alive ──
def _alive():
    while True:
        time.sleep(55)
        try:
            from google.colab import output; output.eval_js('void(0)')
        except: pass
threading.Thread(target=_alive, daemon=True).start()

# ── Dataset ──
train_ds = load_from_disk(f'{DATASET_DIR}/train')
val_ds   = load_from_disk(f'{DATASET_DIR}/val')
print(f'Train:{len(train_ds):,}  Val:{len(val_ds):,}')

# ── GPU check ──
assert torch.cuda.is_available(), 'No GPU! Switch runtime to A100.'
print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB)')

# ── Model ──
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='arabic', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False
model.generation_config.language = 'arabic'
model.generation_config.task     = 'transcribe'
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='arabic', task='transcribe')

wer_metric = evaluate.load('wer')

# ── Collator: on-the-fly feature extraction ──
@dataclass
class Collator:
    processor: Any
    decoder_start_token_id: int = 50257
    _err: int = field(default=0, repr=False)

    def __call__(self, features: List[Dict]) -> Dict:
        ins, labs = [], []
        for f in features:
            try:
                a, sr = sf.read(f['audio_path'], dtype='float32', always_2d=False)
                if a.ndim > 1: a = a.mean(axis=1)
                if sr != 16000:
                    import librosa; a = librosa.resample(a, orig_sr=sr, target_sr=16000)
                if len(a) < 1600: raise ValueError('too short')
                feat = self.processor.feature_extractor(a, sampling_rate=16000).input_features[0]
                ids  = self.processor.tokenizer(f['text']).input_ids
            except Exception as e:
                self._err += 1
                if self._err <= 5: print(f'  ⚠️ {f.get("audio_path","?")}: {e}')
                feat = np.zeros((80,3000), dtype='float32')
                ids  = self.processor.tokenizer('').input_ids
            ins.append({'input_features': feat})
            labs.append({'input_ids': ids})

        batch = self.processor.feature_extractor.pad(ins, return_tensors='pt')
        lb = self.processor.tokenizer.pad(labs, return_tensors='pt')
        lbl = lb['input_ids'].masked_fill(lb.attention_mask.ne(1), -100)
        if (lbl[:,0] == self.decoder_start_token_id).all(): lbl = lbl[:,1:]
        batch['labels'] = lbl
        return batch

def compute_metrics(pred):
    p, l = pred.predictions, pred.label_ids
    l[l==-100] = processor.tokenizer.pad_token_id
    ps = processor.tokenizer.batch_decode(p, skip_special_tokens=True)
    ls = processor.tokenizer.batch_decode(l, skip_special_tokens=True)
    return {'wer': round(wer_metric.compute(predictions=ps, references=ls)*100, 4)}

class AdaptiveEval(TrainerCallback):
    def on_evaluate(self, args, state, control, **kw):
        if state.global_step >= 200 and args.eval_steps != 500:
            args.eval_steps = 500; print('  eval_steps → 500')

class MemClean(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % 200 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache(); gc.collect()

# ── Training args ──
major = int(transformers.__version__.split('.')[0])
kw = dict(
    output_dir=OUTPUT_DIR, per_device_train_batch_size=16,
    per_device_eval_batch_size=8, gradient_accumulation_steps=4,
    learning_rate=1e-5, warmup_steps=500, max_steps=10000,
    gradient_checkpointing=True, fp16=True,
    eval_steps=200, save_strategy='steps', save_steps=500,
    save_total_limit=3, load_best_model_at_end=True,
    metric_for_best_model='wer', greater_is_better=False,
    logging_steps=25, report_to=[], predict_with_generate=True,
    generation_max_length=225, dataloader_num_workers=2,
    dataloader_pin_memory=True, push_to_hub=False,
    remove_unused_columns=False,
)
kw['eval_strategy' if major >= 5 else 'evaluation_strategy'] = 'steps'

tkw = dict(
    model=model, args=Seq2SeqTrainingArguments(**kw),
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=Collator(processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5), AdaptiveEval(), MemClean()],
)
tkw['processing_class' if major >= 5 else 'tokenizer'] = processor.feature_extractor
trainer = Seq2SeqTrainer(**tkw)

# ── Resume from checkpoint if exists ──
ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
resume = ckpts[-1] if ckpts else None
print(f'{"Resuming: "+resume if resume else "New training run"}')
print(f'Batch: 16×4=64 | Checkpoint every 500 steps | Max 10K steps')
print()

trainer.train(resume_from_checkpoint=resume)

# Save
FINAL = f'{OUTPUT_DIR}/final'
trainer.save_model(FINAL)
processor.save_pretrained(FINAL)
print(f'\n{"="*60}\nTRAINING COMPLETE → {FINAL}\nRun Cell 8 to test & push to HuggingFace.\n{"="*60}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — Test + Push to HuggingFace Hub
# ══════════════════════════════════════════════════════════════════════════════
import random, torch
from datasets import load_from_disk
from transformers import pipeline
from huggingface_hub import HfApi, login

FINAL = f'{OUTPUT_DIR}/final'
val_ds = load_from_disk(f'{DATASET_DIR}/val')

# ── Test ──
pipe = pipeline('automatic-speech-recognition', model=FINAL, chunk_length_s=30,
                device=0 if torch.cuda.is_available() else -1,
                generate_kwargs={'language':'arabic','task':'transcribe'})

print('Test (5 validation samples):\n')
for i, s in enumerate(random.sample(list(val_ds), min(5, len(val_ds))), 1):
    try:
        r = pipe(s['audio_path'])
        print(f'[{i}] Expected: {s["text"][:90]}')
        print(f'    Model  : {r["text"][:90]}\n')
    except Exception as e:
        print(f'[{i}] ERROR: {e}\n')

# ── Push to HuggingFace ──
print('='*60)
print(f'Pushing to huggingface.co/{HF_REPO_ID}')
print('='*60 + '\n')

# Login — set HF_TOKEN in Colab secrets, or it will prompt interactively
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

# Model card
card = f"""---
language: ar
license: mit
tags: [whisper, quran, arabic, speech-recognition, fine-tuned]
base_model: openai/whisper-small
pipeline_tag: automatic-speech-recognition
---

# QuranASR — Whisper Fine-tuned for Quran Recitation

Fine-tuned [`openai/whisper-small`](https://huggingface.co/openai/whisper-small) on ~75K Quran recitation samples.

```python
from transformers import pipeline
pipe = pipeline('automatic-speech-recognition', model='{HF_REPO_ID}',
                generate_kwargs={{'language':'arabic','task':'transcribe'}})
print(pipe('quran_audio.mp3')['text'])
```

**Training:** 12 reciters × 6236 verses | batch 64 | lr 1e-5 | A100 GPU

[GitHub](https://github.com/tialkan/QuranASR) · [Tilavet App](https://tilavet.app) · [Author](https://huggingface.co/baristiran)
"""

with open(f'{FINAL}/README.md', 'w') as f: f.write(card)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=FINAL, repo_id=HF_REPO_ID,
                  commit_message='Upload QuranASR whisper-small')

print(f'\n✅ Model live at: https://huggingface.co/{HF_REPO_ID}')